# Learning Development Notebook: myHealth

## Design Philosophy: Concepts + Code = Deep Understanding

This notebook documents my learning journey while building the myHealth web app and LLM inference pipeline. Rather than keeping separate notes (pure text) or only code snippets (no context), this notebook integrates both:

**Why this approach works:**
- **Pure text** lacks executable verification (ideas remain abstract)
- **Pure code** lacks explanation (implementation feels disconnected from why)
- **Concepts + Code together** creates understanding through:
  1. **Explanation**: What is the concept? Why does it matter?
  2. **Implementation**: How do we code this? What patterns emerge?
  3. **Execution**: Run the code. Verify behavior. Test edge cases.
  4. **Reflection**: What gaps remain? How would I explain this to someone else?

---

## Module 1: Foundational Concepts — Pseudonymization & Hashing

### Concept: What is Pseudonymization?

**Definition**: Pseudonymization is the transformation of personal identifiers into a stable, irreversible form that:
- Cannot be reversed without a secret key (deterministic hash)
- Always produces the same output for the same input (enables tracking)
- Appears random (prevents pattern inference)

**Why it matters for myHealth**:
- HIPAA compliance: "Minimize PHI exposure" — we never store raw patient names
- Clinical utility: We still need to track patients across visits (same MRN → same patient ID)
- Auditability: Transformation is deterministic and reproducible (good for compliance)

**Design decision**: Use SHA256 hashing with a SALT value
- Same MRN hashed multiple times = same pseudonym (allows patient tracking)
- Hash is one-way (cannot reverse without brute force)
- Binning age (1958 → "65-74") adds privacy without losing clinical value

In [ ]:
import hashlib
from datetime import datetime

# Concept: Deterministic Hashing for Pseudonymization
# ────────────────────────────────────────────────────────

def pseudonymize_mrn(mrn: str, salt: str = "myhealth-v1") -> str:
    """
    Convert a Medical Record Number (MRN) to a stable pseudonym.
    
    Key insight: SHA256 is deterministic.
    - Same input → Always same output
    - Different input → Different output (with high probability)
    - One-way function → Cannot reverse without brute force
    
    Args:
        mrn: Patient's Medical Record Number (raw PHI)
        salt: Secret value (prevents rainbow table attacks)
    
    Returns:
        Pseudonym ID in format "patient_XXXXX"
    
    HIPAA compliance: Raw MRN never leaves this function as plaintext.
    """
    # Step 1: Combine MRN with SALT to create unique hash
    hash_input = f"{mrn}{salt}".encode('utf-8')
    
    # Step 2: Apply SHA256 hash
    hash_object = hashlib.sha256(hash_input)
    hash_hex = hash_object.hexdigest()  # 64 hex characters
    
    # Step 3: Map hash to 5-digit patient ID (ensures determinism + uniqueness)
    numeric_id = int(hash_hex[:8], 16) % 100000  # Take first 8 hex chars, mod 100000
    
    return f"patient_{numeric_id:05d}"


# Test: Deterministic property
print("Testing deterministic hashing:")
mrn1 = "MRN-123456"
pseudo1 = pseudonymize_mrn(mrn1)
pseudo2 = pseudonymize_mrn(mrn1)  # Same MRN, second time

print(f"First call:  {mrn1} → {pseudo1}")
print(f"Second call: {mrn1} → {pseudo2}")
print(f"Deterministic? {pseudo1 == pseudo2}")  # Should be True

# Test: Different input → Different output
mrn2 = "MRN-654321"
pseudo3 = pseudonymize_mrn(mrn2)
print(f"\nDifferent MRN: {mrn2} → {pseudo3}")
print(f"Different pseudonym? {pseudo1 != pseudo3}")  # Should be True

### Concept: Age Binning for Privacy Preservation

**Problem**: Exact age is re-identifying. Example: "65-year-old female with Type 2 Diabetes living in Portland" can be uniquely identified by linking demographic + diagnosis data.

**Solution**: Bin age into ranges ("65-74" instead of 68). This:
- Reduces precision without losing clinical utility
- Makes re-identification harder (more people match the profile)
- Still allows clinical decision-making (treatment varies by decade, not exact age)

In [ ]:
def bucket_age(dob: str) -> str:
    """
    Convert exact date of birth (PHI) to age bucket (de-identified).
    
    Example: 1958-05-15 → "65-74" (10-year bin, not exact age 68)
    
    Privacy benefit: Harder to re-identify with diagnosis + location
    Clinical benefit: Still clinically meaningful (treatment varies by decade)
    
    Args:
        dob: Date of birth in "YYYY-MM-DD" format
    
    Returns:
        Age bucket string (e.g., "65-74"), or "unknown" if parsing fails
    """
    try:
        birth = datetime.strptime(dob, "%Y-%m-%d")
        today = datetime.now()
        age_years = (today - birth).days // 365  # Approximate (ignores leap years)
        
        # Bin into 10-year ranges
        lower = (age_years // 10) * 10
        upper = lower + 9
        
        return f"{lower}-{upper}"
    except ValueError:
        return "unknown"


# Test: Age binning
test_cases = [
    ("1958-05-15", "65-year-old → '65-74' bucket"),
    ("1985-03-20", "38-year-old → '30-39' bucket"),
    ("2015-12-01", "8-year-old → '0-9' bucket"),
    ("invalid-date", "Invalid date → 'unknown'"),
]

print("Age binning examples:")
for dob, description in test_cases:
    bucket = bucket_age(dob)
    print(f"  {dob:15} → {bucket:10} ({description})")

# Reflection: Why bin by 10 years?
print("\nReflection: Could we use different bin sizes?")
print("  - 5-year bins: More granular, but less privacy (easier to re-identify)")
print("  - 20-year bins: More privacy, but less clinically useful")
print("  - 10 years: Balance between privacy and clinical relevance")


---

## Module 2: Privacy-by-Design Architecture

### Concept: Why Privacy-by-Structure (Not Filtering)?

**Original approach** (filtering-based):
- Raw PHI enters codebase → External (cloud) proxy filters → VPC-constrained model sees "safe" data
- Risk: Filter logic bugs → PHI leaks

**New approach** (structural):
- Raw PHI **never enters** core codebase → Pseudonymization happens upstream → Model sees only structured context
- Risk eliminated: By design, not by filtering

**Implication for developers/agents**:
- You can freely explore, refactor, and improve the code
- Security is **guaranteed by architecture**, not dependent on vigilance
- Raw PHI never appears in the repository (only pseudonymized clinical features)

In [ ]:
# Concept: Privacy Pipeline Boundary
# ────────────────────────────────────────────────────────

# Simulating raw PHI data (what we receive from EHR)
raw_patient_data = {
    "mrn": "MRN-123456",
    "name": "Alice Johnson",  # ← Raw PHI (NEVER leaves this scope)
    "dob": "1958-05-15",      # ← Raw PHI
    "ssn": "123-45-6789",     # ← Raw PHI
    "diagnoses": [{"code": "E11", "desc": "Type 2 Diabetes"}],
    "labs": [{"test": "HbA1c", "value": 8.2}],
}

# Step 1: PSEUDONYMIZE (happens in adapters/pseudonymizer.py)
# ─────────────────────────────────────────────────────────────
pseudonymized_data = {
    "pseudonym_id": pseudonymize_mrn(raw_patient_data["mrn"]),  # ← Hashed
    "age_bucket": bucket_age(raw_patient_data["dob"]),           # ← Binned
    "diagnoses": raw_patient_data["diagnoses"],                 # ← ICD codes (safe)
    "labs": raw_patient_data["labs"],                           # ← Clinical values (safe)
    # Note: NO name, SSN, exact DOB, MRN
    # Note: source_mrn kept for AUDIT ONLY (encrypted logs)
}

# Step 2: Create context safe for LLM
# ───────────────────────────────────────
llm_context = {
    "pseudonym_id": pseudonymized_data["pseudonym_id"],
    "age_bucket": pseudonymized_data["age_bucket"],
    "diagnoses": pseudonymized_data["diagnoses"],
    "labs": pseudonymized_data["labs"],
    # Note: source_mrn is NOT included (audit field stripped)
}

print("Privacy Pipeline Demonstration:")
print("=" * 60)
print(f"\n1. Raw PHI (INTERNAL ONLY):")
print(f"   name: {raw_patient_data['name']}")
print(f"   dob: {raw_patient_data['dob']}")
print(f"   ssn: {raw_patient_data['ssn']}")
print(f"   mrn: {raw_patient_data['mrn']}")

print(f"\n2. Pseudonymized (allowed in core logic):")
print(f"   pseudonym_id: {pseudonymized_data['pseudonym_id']}")
print(f"   age_bucket: {pseudonymized_data['age_bucket']}")
print(f"   diagnoses: {pseudonymized_data['diagnoses']}")

print(f"\n3. LLM Context (safe to send to model):")
print(f"   {llm_context}")
print(f"\n✓ No raw PHI in LLM context")
print(f"✓ All clinical utility preserved")
print(f"✓ Patient can be tracked (deterministic pseudonym)")


---

## Module 3: Citations & Auditability

### Concept: Why Citations Matter

**Problem**: LLM can hallucinate. Example:
- LLM says: "Patient's glucose is 450 mg/dL"
- But the labs show: HbA1c 8.2%, no glucose measurement
- Result: Clinician makes decision based on false data

**Solution**: Every claim must cite a source passage ID:
- Example: "HbA1c 8.2% (passage_42)" — clinician can verify
- Invalid citations are hallucinations (e.g., "passage_999" doesn't exist)
- Audit trail links back to data source

**Implementation**: Store passage IDs alongside data
- Raw patient data → source_passages = ["passage_42", "passage_91"]
- LLM must cite these or be flagged as hallucinating
- Post-LLM verifier checks: all cited passages exist in source list

In [ ]:
import re

def validate_citations(llm_output: str, valid_passage_ids: list[str]) -> dict:
    """
    Check if LLM output citations are valid (not hallucinated).
    
    Concept: Citations link claims to verifiable sources.
    - Valid: "HbA1c 8.2% (passage_42)" where passage_42 exists
    - Invalid: "HbA1c 8.2% (passage_999)" where passage_999 doesn't exist
    
    Args:
        llm_output: Text from LLM
        valid_passage_ids: List of passage IDs that actually exist in source data
    
    Returns:
        Dict with:
        - cited_passages: Set of passage IDs cited in output
        - invalid_citations: Passages cited but don't exist (hallucinations)
        - valid_citations: Passages cited that do exist
        - has_citations: Whether output has any citations
    """
    # Extract all passages cited in format "passage_NNN"
    citation_pattern = r"passage_(\d+)"
    cited = set(f"passage_{num}" for num in re.findall(citation_pattern, llm_output))
    
    # Check which are valid
    valid_cited = set(p for p in cited if p in valid_passage_ids)
    invalid_cited = set(p for p in cited if p not in valid_passage_ids)
    
    return {
        "cited_passages": cited,
        "valid_citations": valid_cited,
        "invalid_citations": invalid_cited,
        "has_citations": len(cited) > 0,
    }


# Test: Citation validation
print("Citation Validation Example:")
print("=" * 60)

# Available data sources
source_passages = ["passage_42", "passage_91", "passage_105"]

# LLM output 1: Correct citations
llm_output_1 = """
Patient has elevated HbA1c at 8.2% (passage_42). 
Blood pressure is 142/90 mmHg (passage_91).
Recent labs show good kidney function (passage_105).
"""

result_1 = validate_citations(llm_output_1, source_passages)
print("\nTest 1: Correct citations")
print(f"  Output: {llm_output_1[:50]}...")
print(f"  Valid citations: {result_1['valid_citations']}")
print(f"  Invalid citations: {result_1['invalid_citations']}")
print(f"  ✓ All citations valid" if not result_1['invalid_citations'] else "  ✗ Hallucinations detected!")

# LLM output 2: Hallucinated citation
llm_output_2 = """
Patient has critically elevated glucose at 450 mg/dL (passage_999).
This requires immediate intervention (passage_999).
"""

result_2 = validate_citations(llm_output_2, source_passages)
print("\nTest 2: Hallucinated citations")
print(f"  Output: {llm_output_2[:50]}...")
print(f"  Valid citations: {result_2['valid_citations']}")
print(f"  Invalid citations: {result_2['invalid_citations']}")
print(f"  ✗ Hallucinations detected!" if result_2['invalid_citations'] else "  ✓ All citations valid")

# LLM output 3: No citations (also a problem!)
llm_output_3 = "Patient seems to have high blood sugar based on general knowledge."

result_3 = validate_citations(llm_output_3, source_passages)
print("\nTest 3: Missing citations")
print(f"  Output: {llm_output_3}")
print(f"  Has citations: {result_3['has_citations']}")
print(f"  ✗ Missing citations" if not result_3['has_citations'] else "  ✓ Has citations")


---

## Module 4: Post-LLM Validation (Rule Layer)

### Concept: Don't Trust the Model Blindly

**Why**: LLMs can:
- Hallucinate (invent data that wasn't provided)
- Make medically implausible claims (HR 500 bpm, HbA1c 15%)
- Suggest inappropriate actions ("prescribe immediately" vs "discuss with provider")
- Leak PHI despite instructions not to

**Solution**: Rule-based validation after LLM:
1. **PHI detection**: Regex scan for emails, SSNs, full dates, names
2. **Plausibility checks**: Lab/vital values in medical ranges
3. **Safety checks**: Forbid prescriptive advice
4. **Structure checks**: Require citations, confidence levels, headings

**Effect**: Confidence score reduced by violations, flags for human review

In [ ]:
import re

def validate_medical_plausibility(text: str) -> list[str]:
    """
    Check if text contains medically implausible claims.
    
    Concept: LLM can hallucinate medical values that sound plausible but aren't.
    Example: "Heart rate 500 bpm" (impossible; human HR max ~200)
    
    Returns list of violations found.
    """
    violations = []
    
    # Pattern: "NNN bpm" heart rate claims
    hr_pattern = r"(\d+)\s*bpm"
    for match in re.finditer(hr_pattern, text):
        hr = int(match.group(1))
        if hr < 40 or hr > 200:
            violations.append(f"Invalid HR {hr} bpm (normal: 40-200)")
    
    # Pattern: "NN%" HbA1c claims
    hba1c_pattern = r"(?:HbA1c|A1c)\s*([0-9.]+)%"
    for match in re.finditer(hba1c_pattern, text):
        hba1c = float(match.group(1))
        if hba1c < 4 or hba1c > 14:
            violations.append(f"Invalid HbA1c {hba1c}% (normal: 4-14%)")
    
    return violations


# Test: Medical plausibility
print("Testing medical plausibility validation:")
test_outputs = [
    ("Heart rate 85 bpm", "Valid HR"),
    ("Heart rate 500 bpm", "Implausible HR (hallucination)"),
    ("HbA1c 8.2%", "Valid HbA1c"),
    ("HbA1c 25%", "Implausible HbA1c (hallucination)"),
]

for text, description in test_outputs:
    violations = validate_medical_plausibility(text)
    status = "✓ PASS" if not violations else f"✗ FAIL: {violations}"
    print(f"  '{text}' → {status}")


---

## Module 5: Error Handling & Audit Trails

### Concept: Never Log Raw PHI; Always Log What Went Wrong

When something fails in the pipeline, you need to:
1. **Log enough detail** to debug (request ID, timestamp, which stage failed)
2. **Log zero raw PHI** (never log patient name, SSN, medical details)
3. **Log the action taken** (rejected output, escalated to human, retried)

### Design Pattern: Pseudonym-Based Error Logging

Instead of:
```
ERROR: Jane Doe (MRN-123) has invalid HbA1c 25% (HIPAA violation!)
```

Do:
```
ERROR: patient_00042, validation_failed, reason=medical_plausibility, field=HbA1c, invalid_value=25%, source=bedrock_response, action=rejected, escalated_to=clinician_review
```

Key differences:
- **patient_XXXXX** instead of name (safe for logs)
- **Field names** instead of values (HbA1c, not "25%")
- **Reason codes** instead of descriptions (medical_plausibility, not "glucose value impossible")
- **Action taken** explicit (rejected, escalated, retry)
- **Audit trail intact** (CloudWatch logs linked to request ID)

---

## Module 6: Data Structures for Privacy

### Concept: Careful Design Prevents Accidental PHI Exposure

The way you structure data determines what information is accessible at each stage. Good design makes it **impossible** to accidentally leak PHI.

**Bad Structure** (raw data everywhere):
```python
patient = {
    "name": "John Doe",           # PHI - accessible to everyone
    "mrn": "MRN-123456",           # PHI - accessible to everyone
    "dob": "1960-05-15",           # PHI - accessible to everyone
    "heart_rate": 72,              # Fine
    "hba1c": 8.2,                  # Fine
}
# Problem: If you serialize this JSON and send to LLM, you leak PHI
```

**Good Structure** (PHI gated at boundary):
```python
# Raw data (internal only, stays in adapters/)
raw_patient = {
    "mrn": "MRN-123456",
    "name": "John Doe",
    "dob": "1960-05-15",
    "heart_rate": 72,
    "hba1c": 8.2,
}

# Pseudonymized data (safe for LLM, storage, logs)
pseudonymized_patient = {
    "pseudonym_id": "patient_00001",  # Hash of MRN; no reverse lookup
    "age_bucket": "60-69",             # Bucketed; can't re-identify by exact age
    "clinical_features": {
        "heart_rate": 72,
        "hba1c": 8.2,
        "icd_codes": ["E10", "I10"],   # ICD codes only; no descriptions
    },
    "source": "patient_00001:passage_42",  # Audit trail; safe ID
}
# Benefit: Can serialize this to JSON safely; no raw PHI present
```

### Key Design Patterns for Privacy Structures

**Pattern 1: Sealed Raw Data (Adapter Boundary)**
- Raw PHI exists only in adapters (e.g., `adapters/pseudonymizer.py`)
- Once pseudonymized, raw data is forgotten
- Function signature makes this explicit

**Pattern 2: Immutable Pseudonym Mapping**
- Store pseudonym ID → hash function only (not hash → raw)
- Enables auditing ("which patient_XXXXX is this?") but prevents reverse lookup
- SHA256 is one-way: can't recover MRN from patient_00001

**Pattern 3: Source Attribution (Passage IDs)**
- Every claim must reference where it came from: `passage_42`, `passage_43`
- Enables citation validation; prevents hallucination
- Doesn't expose raw data; just pointer to source

**Pattern 4: Confidence as Data Quality Signal**
- Attach confidence score to outputs (0.0-1.0)
- Reflects uncertainty, not false precision
- Low confidence (<0.7) triggers human review

---

## Module 7: Core Algorithms & Patterns

### Concept: The Four-Stage Pipeline

myHealth's core strength is a **deterministic, auditable pipeline** that makes privacy a property of architecture, not implementation.

**Stage 1: Pseudonymize**
- Input: Raw PHI (name, MRN, DOB, health data)
- Process: Hash MRN → patient_XXXXX; bucket age; extract codes
- Output: Pseudonymized clinical context (no PHI)
- Key: Deterministic; same MRN always → same pseudonym

**Stage 2: Build Prompt**
- Input: Pseudonymized context + user question
- Process: Structure as clinical template; attach citations
- Output: LLM-safe prompt (structured, with guardrails)
- Key: Citations required; safety disclaimers embedded

**Stage 3: LLM Inference**
- Input: Prompt (no raw PHI)
- Process: Call Bedrock (inside VPC); get response
- Output: LLM output (potentially flawed)
- Key: All comms stay inside AWS VPC; zero internet exposure

**Stage 4: Validate & Redact**
- Input: LLM output + source passage IDs
- Process: Check citations, medical plausibility, PHI leakage, safety
- Output: Verified output (confidence score) OR rejected with reason
- Key: Rule-based; deterministic; auditable

---

## Module 8: Best Practices & Reflection

### Best Practice 1: Privacy by Design, Not Filtering

**Wrong approach:**
```
Raw PHI → LLM → Catch leakage in output (filtering)
Problem: Leakage already happened; hard to detect all cases
```

**Right approach:**
```
Raw PHI → Pseudonymize (upstream) → LLM (never sees raw PHI)
Benefit: No leakage possible; design guarantees safety
```

**Lesson**: Build systems where the architecture itself prevents the problem. Filtering is a last resort, not a security strategy.

### Best Practice 2: Deterministic Hashing for Patient Tracking

**Why not random IDs?**
- Random: Different ID each time → can't track same patient across requests
- Deterministic: Same MRN → same ID → can track + audit

**Why one-way hash?**
- Hash is irreversible: given patient_00001, can't recover MRN
- Still deterministic: MRN-123 → patient_00001 (always)
- Combined with audit trail: who accessed patient_00001, when, why

**Lesson**: Deterministic hashing enables functionality (patient tracking) AND privacy (irreversibility) simultaneously.

### Best Practice 3: Confidence Scores Reflect Uncertainty

**Wrong approach:**
```
Verifier says: "Output is valid" (boolean)
Problem: False confidence; user assumes it's trusted
```

**Right approach:**
```
Verifier says: "Output has 0.85 confidence; 1 citation invalid, score reduced by 0.1"
Benefit: Uncertainty explicit; human knows to review edge cases
```

**Lesson**: Transparency about confidence helps humans make better decisions.

### Best Practice 4: Rule-Based Validation Catches What Prompting Misses

**Example: LLM hallucinates impossible medical values**
```
Prompt says: "Only report plausible values"
LLM output: "Heart rate 500 bpm"
Problem: Prompt didn't work; model still hallucinated

Solution: Rule layer checks HR is 40-200 bpm → rejects 500
Lesson: Prompts guide behavior; rules enforce it
```

### Best Practice 5: Audit Trails Enable Accountability

**What to log (safe):**
- Request ID, timestamp, patient pseudonym, action (approved/rejected)
- Reason code (medical_implausibility, phi_leakage, etc.)
- Confidence score, sources cited

**What NOT to log (PHI):**
- Patient names, MRNs, full DOBs
- Medical values, diagnoses, medications
- Email addresses, phone numbers, addresses

**Lesson**: Audit trails protect both patients (evidence of proper handling) and organization (defense against liability).

### Reflection Questions

1. **Privacy by Design**: If you removed the pseudonymizer, could LLM still cause harm?
2. **Determinism**: Why is it better that MRN-123 always → patient_00001, instead of a random ID?
3. **Citations**: What happens if LLM can't cite its sources?
4. **Confidence**: If confidence is 0.95, does that mean the output is correct?
5. **Audit Trail**: Why log pseudonym_id but not patient name?
6. **Edge Case**: What if pseudonymizer fails to detect raw PHI?

---

**Final Synthesis**: The myHealth pipeline works because it layers multiple privacy techniques:
- **Pseudonymization** (upstream) prevents raw PHI exposure
- **Citations** (prompt layer) enable verification
- **Validation rules** (downstream) catch hallucinations & violations
- **Confidence scoring** (transparency) allows human judgment
- **Audit logging** (safety net) provides accountability

No single technique is foolproof. Together, they make the system robust.

In [ ]:
# Final Synthesis: Full Pipeline Execution with Reflection
import re
from datetime import datetime, date

print("=" * 70)
print("FINAL SYNTHESIS: Privacy-Preserving LLM Pipeline")
print("=" * 70)

# Simulate a complete flow
class PipelineExecutor:
    """
    Execute the full privacy pipeline and log results.
    Shows how all modules work together.
    """
    
    def __init__(self):
        self.salt = "myhealth_secret_2025"
    
    def execute(self, mrn: str, dob: date, query: str):
        """Execute pipeline end-to-end."""
        print(f"\n[PIPELINE EXECUTION]")
        print(f"  Input MRN: {mrn} (PHI - will be pseudonymized)")
        print(f"  Input DOB: {dob} (PHI - will be binned)")
        print(f"  Query: {query}")
        
        print(f"\n  Stage 1: PSEUDONYMIZE")
        pseudonym = pseudonymize_mrn(mrn)
        age_bucket = bucket_age(dob.isoformat())
        print(f"    MRN {mrn} → {pseudonym}")
        print(f"    DOB {dob} → age bucket {age_bucket}")
        print(f"    ✓ Raw PHI eliminated; pseudonymized context ready for LLM")
        
        print(f"\n  Stage 2: BUILD PROMPT")
        llm_prompt = (
            f"Patient {pseudonym} (age {age_bucket}) asks: {query}\n"
            f"Provide response with citations to passage IDs (passage_NNN).\n"
            f"Do NOT use exact medical values without citing source.\n"
            f"Do NOT recommend immediate prescription; suggest discussion with provider."
        )
        print(f"    Prompt (LLM-safe): '{llm_prompt[:80]}...'")
        print(f"    ✓ Pseudonymized context; safety guardrails embedded")
        
        print(f"\n  Stage 3: LLM INFERENCE")
        good_output = (
            f"{pseudonym} has concerning trend in vitals (passage_42). "
            f"Heart rate 85 bpm (passage_42). Discuss with provider about next steps."
        )
        bad_output = (
            f"Patient {mrn} named John has possible issue. Heart rate 500 bpm. "
            f"Prescribe medication immediately (passage_999)."
        )
        print(f"    Good output: '{good_output[:60]}...'")
        print(f"    Bad output: '{bad_output[:60]}...'")
        
        print(f"\n  Stage 4: VALIDATE & REDACT")
        valid_sources = ["passage_42", "passage_43"]
        
        print(f"    Good output validation:")
        good_citations = validate_citations(good_output, valid_sources)
        good_medical = validate_medical_plausibility(good_output)
        good_confidence = 0.95 if not good_citations['invalid_citations'] and not good_medical else 0.70
        print(f"      Citations valid: {len(good_citations['valid_citations']) > 0}")
        print(f"      Medical plausible: {len(good_medical) == 0}")
        print(f"      Confidence: {good_confidence:.2f} → {'APPROVED' if good_confidence >= 0.7 else 'ESCALATE'}")
        
        print(f"    Bad output validation:")
        bad_citations = validate_citations(bad_output, valid_sources)
        bad_medical = validate_medical_plausibility(bad_output)
        bad_phi = any(x in bad_output for x in ['MRN', 'John', 'named'])
        bad_confidence = 0.2 if bad_phi or bad_citations['invalid_citations'] or bad_medical else 0.7
        print(f"      Citations valid: {len(bad_citations['valid_citations']) > 0} (passage_999 invalid)")
        print(f"      Medical plausible: {len(bad_medical) == 0} {bad_medical}")
        print(f"      PHI leakage: {bad_phi}")
        print(f"      Confidence: {bad_confidence:.2f} → {'APPROVED' if bad_confidence >= 0.7 else 'ESCALATE'}")
        
        print(f"\n  Stage 5: AUDIT LOGGING (PHI-safe)")
        audit_entry = (
            f"request_id=req_123 pseudonym={pseudonym} "
            f"action=approved confidence={good_confidence:.2f} "
            f"timestamp={datetime.utcnow().isoformat()}"
        )
        print(f"    {audit_entry}")
        print(f"    ✓ Audit trail logged; no raw PHI in log")

# Execute
executor = PipelineExecutor()
executor.execute(
    mrn="MRN-987654",
    dob=date(1962, 3, 20),
    query="I've been feeling fatigued lately; should I be concerned?"
)

print("\n" + "=" * 70)
print("KEY INSIGHTS FROM THIS EXECUTION:")
print("=" * 70)
print("""
1. INPUT → PSEUDONYMIZE
   Raw PHI (MRN, name, DOB) stays in adapter only.
   Pseudonymized ID (patient_00001) is what LLM sees.

2. PSEUDONYMIZE → BUILD PROMPT
   Prompt contains zero identifying information.
   Safety guardrails embedded (require citations, forbid prescriptions).

3. BUILD PROMPT → INFERENCE
   LLM operates inside VPC with pseudonymized data only.
   No raw PHI reaches model; no public internet exposure.

4. INFERENCE → VALIDATE
   Even if LLM hallucinates (500 bpm, leaks PHI), validation catches it.
   Rule layer is the safety net when prompting fails.

5. VALIDATE → AUDIT
   Confidence score reflects uncertainty.
   Audit trail uses pseudonyms (safe for logs).
   Human review for outputs < 0.7 confidence.

6. PRIVACY GUARANTEE
   This architecture makes PHI leakage structurally impossible.
   Security emerges from design, not from filtering.
   Agents can safely work on this code without risk of exposure.
""")

print("=" * 70)
print("✓✓✓ Learning Complete! You now understand privacy-preserving LLM pipelines.")
print("=" * 70)
print("✔")
print("\u2713")("✔")